# Підсумкова статистика курсової роботи

**Тема:** Аналіз технологічних навичок на українському IT-ринку: автоматизований збір, дедуплікація та регресійне моделювання впливу на заробітну плату

**Призначення цього ноутбука:** єдина точка перевірки всіх кількісних результатів, на які посилається текст звіту.

**Вхідні файли:**
- `step2_master_vacancies_clustered.parquet` — корпус після дедуплікації (parents + children)
- `step3_vacancies_with_skills.parquet` — корпус після витягу скілів (тільки parents)
- `regression_results.csv` — коефіцієнти регресії з notebook 04
- `skillgap_top_pairs.csv` — топ пар skill-gap з notebook 05
- `skillgap_recommendations.csv` — рекомендації skill-gap з notebook 05

## 1. Налаштування

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# Адаптуйте під своє середовище (Kaggle / Colab / локально) та замініть на свої шляхи
STEP2_PATH = 'шлях/до/step2_master_vacancies_clustered.parquet'
STEP3_PATH = 'шлях/до/step3_vacancies_with_skills.parquet'
REGRESSION_CSV = 'шлях/до/regression_results.csv'
SKILLGAP_PAIRS_CSV = 'шлях/до/skillgap_top_pairs.csv'
SKILLGAP_RECS_CSV = 'шлях/до/skillgap_recommendations.csv'

step2 = pd.read_parquet(STEP2_PATH)
step3 = pd.read_parquet(STEP3_PATH)
regression = pd.read_csv(REGRESSION_CSV)
pairs = pd.read_csv(SKILLGAP_PAIRS_CSV)
recs = pd.read_csv(SKILLGAP_RECS_CSV)

print('Усі файли завантажено успішно')
print(f'  step2:      {len(step2):,} рядків')
print(f'  step3:      {len(step3):,} рядків')
print(f'  regression: {len(regression):,} коефіцієнтів-скілів')
print(f'  pairs:      {len(pairs):,} асоціативних пар')

Усі файли завантажено успішно
  step2:      46,227 рядків
  step3:      27,720 рядків
  regression: 48 коефіцієнтів-скілів
  pairs:      951 асоціативних пар


## 2. Розмір корпусу даних — підрозділ 3.2, 3.7.1

Двоетапна дедуплікація: 69 949 сирих → 27 720 батьківських + 18 507 дочірніх вакансій.

In [2]:
# Сирих записів до дедуплікації — записано історично у звіті
RAW_RECORDS = 69_949

# Після етапу 1 (точна агрегація за global_id) — це поточний розмір step2
after_step1 = len(step2)

# Після етапу 2 (семантична кластеризація)
parents = step2[step2['is_parent'] == True]
children = step2[step2['is_parent'] == False]
n_parents = len(parents)
n_children = len(children)

# Зниження потужності
reduction_pct = (1 - n_parents / RAW_RECORDS) * 100

print('=== ДЕДУПЛІКАЦІЯ ===')
print(f'  Сирих записів:                       {RAW_RECORDS:,}')
print(f'  Після етапу 1 (точна агрегація):     {after_step1:,}')
print(f'  Після етапу 2 (семантична):          {n_parents:,} parents + {n_children:,} children')
print(f'  Зниження потужності:                 {reduction_pct:.1f}%')

=== ДЕДУПЛІКАЦІЯ ===
  Сирих записів:                       69,949
  Після етапу 1 (точна агрегація):     46,227
  Після етапу 2 (семантична):          27,720 parents + 18,507 children
  Зниження потужності:                 60.4%


## 3. Період збору даних

In [3]:
# Беремо найранішу і найпізнішу дату з seen_dates
all_dates = []
for sd in step2['seen_dates'].dropna():
    if isinstance(sd, (list, np.ndarray)) and len(sd) > 0:
        all_dates.extend(sd)

if all_dates:
    all_dates = pd.to_datetime(all_dates, errors='coerce')
    all_dates = all_dates[pd.notna(all_dates)]
    min_date = all_dates.min()
    max_date = all_dates.max()
    print(f'  Період збору:  з {min_date.strftime("%d.%m.%Y")} по {max_date.strftime("%d.%m.%Y")}')
    print(f'  Тривалість:    {(max_date - min_date).days} днів')

# Розподіл за джерелами
print(f'\n  За джерелами (parents):')
for src, cnt in parents['source'].value_counts().items():
    print(f'    {src:<10} {cnt:,}')

  Період збору:  з 05.03.2025 по 12.05.2026
  Тривалість:    433 днів

  За джерелами (parents):
    djinni     19,112
    dou        8,608


## 4. Зарплати — основа для регресії

**Ключові цифри:**
- 4 498 батьківських вакансій з public_salary_min
- 4 497 — з обома (min і max), використовуються для регресії
- 2 618 — Djinni-вакансії, що увійшли в фінальну вибірку

In [4]:
# Зарплати у parents
n_with_salary_min = parents['public_salary_min'].notna().sum()
n_with_salary_max = parents['public_salary_max'].notna().sum()
n_with_both = (parents['public_salary_min'].notna() & parents['public_salary_max'].notna()).sum()
n_with_either = (parents['public_salary_min'].notna() | parents['public_salary_max'].notna()).sum()

print('=== ЗАРПЛАТИ У PARENTS ===')
print(f'  З public_salary_min:         {n_with_salary_min:,}')
print(f'  З public_salary_max:         {n_with_salary_max:,}')
print(f'  З обома (min І max):         {n_with_both:,}  ← використовуються для регресії')
print(f'  З хоча б одним (min АБО max): {n_with_either:,}')

# Розподіл за джерелами серед parents з salary
with_salary = parents[parents['public_salary_min'].notna() & parents['public_salary_max'].notna()].copy()
with_salary['salary_mid'] = (with_salary['public_salary_min'] + with_salary['public_salary_max']) / 2

print(f'\n  Серед них за джерелами:')
for src, cnt in with_salary['source'].value_counts().items():
    print(f'    {src:<10} {cnt:,}')

# Фільтр на робочий діапазон
filtered = with_salary[(with_salary['salary_mid'] >= 200) & (with_salary['salary_mid'] <= 20000)]
djinni_filtered = filtered[filtered['source'] == 'djinni']

print(f'\n  Після фільтрації [200; 20 000] USD/міс.:')
print(f'    Загалом:        {len(filtered):,}')
print(f'    Тільки Djinni:  {len(djinni_filtered):,}  ← N у регресії')

=== ЗАРПЛАТИ У PARENTS ===
  З public_salary_min:         4,498
  З public_salary_max:         4,497
  З обома (min І max):         4,497  ← використовуються для регресії
  З хоча б одним (min АБО max): 4,498

  Серед них за джерелами:
    djinni     2,817
    dou        1,680

  Після фільтрації [200; 20 000] USD/міс.:
    Загалом:        4,284
    Тільки Djinni:  2,618  ← N у регресії


In [5]:
# Описова статистика зарплат
print('=== РОЗПОДІЛ ЗАРПЛАТ (USD/міс., після фільтру) ===')
print(djinni_filtered['salary_mid'].describe().round(0).to_string())

# Перцентилі
print(f'\n  10-й перцентиль:  {djinni_filtered["salary_mid"].quantile(0.10):.0f} USD')
print(f'  25-й перцентиль:  {djinni_filtered["salary_mid"].quantile(0.25):.0f} USD')
print(f'  Медіана:          {djinni_filtered["salary_mid"].median():.0f} USD')
print(f'  75-й перцентиль:  {djinni_filtered["salary_mid"].quantile(0.75):.0f} USD')
print(f'  90-й перцентиль:  {djinni_filtered["salary_mid"].quantile(0.90):.0f} USD')

=== РОЗПОДІЛ ЗАРПЛАТ (USD/міс., після фільтру) ===
count     2618.0
mean      2617.0
std       2081.0
min        200.0
25%       1100.0
50%       2000.0
75%       3500.0
max      17500.0

  10-й перцентиль:  650 USD
  25-й перцентиль:  1100 USD
  Медіана:          2000 USD
  75-й перцентиль:  3500 USD
  90-й перцентиль:  5283 USD


## 5. Скіли — підрозділ 3.7.1

**Ключові цифри:**
- 199 959 загальних згадок
- 34 780 унікальних канонічних назв
- 19 797 вакансій (71,4%) з принаймні одним скілом
- 7,21 — середня кількість скілів на вакансію

In [6]:
# Підрахунок скілів по всьому корпусу step3
skill_counter = Counter()
for s_list in step3['skills']:
    if isinstance(s_list, (list, np.ndarray)):
        skill_counter.update(s_list)

n_with_skills = (step3['skills_count'] >= 1).sum()
n_with_2plus = (step3['skills_count'] >= 2).sum()
mean_skills = step3['skills_count'].mean()
median_skills = step3['skills_count'].median()

print('=== СКІЛИ У КОРПУСІ ===')
print(f'  Загальна кількість згадок:           {sum(skill_counter.values()):,}')
print(f'  Унікальних канонічних назв:          {len(skill_counter):,}')
print(f'  Вакансій з ≥1 скілом:                {n_with_skills:,}  ({n_with_skills/len(step3)*100:.1f}%)')
print(f'  Вакансій з ≥2 скілами (для skill-gap): {n_with_2plus:,}')
print(f'  Середня кількість скілів/вакансія:   {mean_skills:.2f}')
print(f'  Медіана:                             {median_skills:.0f}')

=== СКІЛИ У КОРПУСІ ===
  Загальна кількість згадок:           199,959
  Унікальних канонічних назв:          34,780
  Вакансій з ≥1 скілом:                19,797  (71.4%)
  Вакансій з ≥2 скілами (для skill-gap): 19,472
  Середня кількість скілів/вакансія:   7.21
  Медіана:                             6


In [7]:
# Топ-10 скілів — для таблиці 3.1
print('=== ТАБЛИЦЯ 3.1 — ТОП-10 НАВИЧОК ===')
print(f'{"№":<4}{"Навичка":<20}{"Кількість згадок":>18}')
print('-' * 42)
for i, (skill, cnt) in enumerate(skill_counter.most_common(10), 1):
    print(f'{i:<4}{skill:<20}{cnt:>18,}')

=== ТАБЛИЦЯ 3.1 — ТОП-10 НАВИЧОК ===
№   Навичка               Кількість згадок
------------------------------------------
1   Python                           7,268
2   Docker                           4,268
3   AWS                              4,178
4   Kubernetes                       4,154
5   PostgreSQL                       3,962
6   SQL                              3,382
7   Agile                            2,923
8   Git                              2,744
9   CI/CD                            2,543
10  Scrum                            2,415


In [8]:
print('=== РОЗШИРЕНИЙ ТОП-25 (для довідки) ===')
for i, (skill, cnt) in enumerate(skill_counter.most_common(25), 1):
    print(f'  {i:>3}. {skill:<25} {cnt:,}')

=== РОЗШИРЕНИЙ ТОП-25 (для довідки) ===
    1. Python                    7,268
    2. Docker                    4,268
    3. AWS                       4,178
    4. Kubernetes                4,154
    5. PostgreSQL                3,962
    6. SQL                       3,382
    7. Agile                     2,923
    8. Git                       2,744
    9. CI/CD                     2,543
   10. Scrum                     2,415
   11. REST                      1,828
   12. Azure                     1,791
   13. React                     1,699
   14. JavaScript                1,689
   15. TypeScript                1,649
   16. Terraform                 1,609
   17. GCP                       1,602
   18. Node.js                   1,438
   19. Redis                     1,406
   20. Linux                     1,326
   21. Jira                      1,261
   22. MySQL                     1,180
   23. Grafana                   1,149
   24. FastAPI                   1,136
   25. MongoDB          

## 6. Покриття за категоріями

Перевірка валідності: технологічні категорії мають високе покриття скілами, нетехнічні (Sales, HR, Design) — низьке. Це підтверджує, що LLM працює адекватно.

In [9]:
coverage = step3.groupby('category').agg(
    total=('skills_count', 'size'),
    with_skills=('skills_count', lambda x: (x >= 1).sum()),
).assign(coverage_pct=lambda d: (d['with_skills'] / d['total'] * 100).round(1))
coverage = coverage.sort_values('total', ascending=False).head(20)

print('=== ПОКРИТТЯ СКІЛАМИ ПО ТОП-20 КАТЕГОРІЯХ ===')
print(f'{"Категорія":<25}{"Усього":>10}{"З скілами":>12}{"%":>8}')
print('-' * 55)
for cat, row in coverage.iterrows():
    print(f'{cat:<25}{row["total"]:>10,}{row["with_skills"]:>12,}{row["coverage_pct"]:>7.1f}%')

=== ПОКРИТТЯ СКІЛАМИ ПО ТОП-20 КАТЕГОРІЯХ ===
Категорія                    Усього   З скілами       %
-------------------------------------------------------
Marketing                   3,200.0     1,291.0   40.3%
Python                      2,249.0     2,240.0   99.6%
DevOps                      2,024.0     1,992.0   98.4%
Data Science                1,921.0     1,851.0   96.4%
Sales                       1,424.0       295.0   20.7%
Data Engineer               1,114.0     1,101.0   98.8%
Project Manager               839.0       469.0   55.9%
Support                       828.0       392.0   47.3%
Other                         726.0       435.0   59.9%
Design                        682.0       153.0   22.4%
Fullstack                     615.0       613.0   99.7%
HR                            599.0       175.0   29.2%
Golang                        552.0       548.0   99.3%
QA                            531.0       504.0   94.9%
Product Manager               477.0       322.0   67.5%
SE

## 7. Регресія — підрозділ 3.7.2

**Ключові цифри:**
- N = 2 618 (Djinni-only)
- R² = 0,5865, Adj R² = 0,5743
- F = 48,06, p < 0,001
- 48 індикаторних змінних навичок
- 9 статистично значущих (6 позитивних + 3 негативних)

In [10]:
# Поточні налаштування регресії — для довідки
print('=== ПАРАМЕТРИ РЕГРЕСІЇ ===')
print(f'  Залежна змінна:        ln(salary_mid)')
print(f'  Вибірка:               Djinni-only')
print(f'  Фільтр зарплати:       [200; 20 000] USD/міс.')
print(f'  Поріг частоти скілу:   ≥ 50 у вибірці')
print(f'  N (спостережень):      {len(djinni_filtered):,}')
print(f'  Скілів у моделі:       {len(regression):,}')

# Зведена статистика — з регресійних результатів
n_sig = (regression['p_value'] < 0.05).sum()
n_sig_pos = ((regression['p_value'] < 0.05) & (regression['coef'] > 0)).sum()
n_sig_neg = ((regression['p_value'] < 0.05) & (regression['coef'] < 0)).sum()

print(f'\n  Статистично значущих ефектів (p < 0,05):')
print(f'    Усього:    {n_sig}')
print(f'    Позитивні: {n_sig_pos}')
print(f'    Негативні: {n_sig_neg}')

=== ПАРАМЕТРИ РЕГРЕСІЇ ===
  Залежна змінна:        ln(salary_mid)
  Вибірка:               Djinni-only
  Фільтр зарплати:       [200; 20 000] USD/міс.
  Поріг частоти скілу:   ≥ 50 у вибірці
  N (спостережень):      2,618
  Скілів у моделі:       48

  Статистично значущих ефектів (p < 0,05):
    Усього:    9
    Позитивні: 6
    Негативні: 3


In [11]:
# Таблиця 3.2 — значущі скіли
significant = regression[regression['p_value'] < 0.05].copy()
significant = significant.sort_values('coef', ascending=False)

print('=== ТАБЛИЦЯ 3.2 — ЗНАЧУЩІ ВПЛИВИ СКІЛІВ ===')
print(f'{"Скіл":<20}{"β":>10}{"%":>10}{"p-value":>12}')
print('-' * 52)
for _, r in significant.iterrows():
    sign = '+' if r['coef'] > 0 else ''
    print(f'{r["skill"]:<20}{sign}{r["coef"]:>9.3f}{sign}{r["pct_effect"]:>9.1f}{r["p_value"]:>12.4f}')

=== ТАБЛИЦЯ 3.2 — ЗНАЧУЩІ ВПЛИВИ СКІЛІВ ===
Скіл                         β         %     p-value
----------------------------------------------------
TensorFlow          +    0.238+     26.9      0.0465
Java                +    0.190+     21.0      0.0054
C++                 +    0.154+     16.6      0.0260
Python              +    0.131+     14.0      0.0009
Kafka               +    0.130+     13.9      0.0469
Kubernetes          +    0.116+     12.3      0.0045
Git                    -0.105    -10.0      0.0079
Django                 -0.147    -13.6      0.0294
Bash                   -0.160    -14.8      0.0299


In [12]:
import os
summary_path = '/kaggle/input/datasets/danlnustudent/tech-skills-results/regression_summary.txt'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        content = f.read()
    print('=== ФРАГМЕНТ STATSMODELS SUMMARY ===')
    # Показуємо тільки ключові рядки
    for line in content.split('\n')[:15]:
        print(line)
    print('... (повний звіт у файлі regression_summary.txt)')

=== ФРАГМЕНТ STATSMODELS SUMMARY ===
                            OLS Regression Results                            
Dep. Variable:             log_salary   R-squared:                       0.586
Model:                            OLS   Adj. R-squared:                  0.574
Method:                 Least Squares   F-statistic:                     48.06
Date:                Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                        16:29:15   Log-Likelihood:                -1947.3
No. Observations:                2618   AIC:                             4047.
Df Residuals:                    2542   BIC:                             4493.
Df Model:                          75                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

## 8. Skill-gap аналіз — підрозділ 3.7.3

**Ключові цифри:**
- 19 472 вакансії з ≥ 2 скілами
- 203 популярні скіли (≥ 100 згадок)
- 951 пара з support ≥ 0,5%

In [13]:
# Параметри skill-gap аналізу
popular_threshold = 100
popular_skills = sum(1 for c in skill_counter.values() if c >= popular_threshold)

print('=== ПАРАМЕТРИ SKILL-GAP АНАЛІЗУ ===')
print(f'  Вибірка (вакансій з ≥2 скілами):     {n_with_2plus:,}')
print(f'  Поріг "популярний" скіл (≥):         {popular_threshold} згадок')
print(f'  Кількість популярних скілів:         {popular_skills}')
print(f'  Поріг support пари:                  ≥ 0,5%')
print(f'  Кількість пар у результатах:         {len(pairs):,}')

=== ПАРАМЕТРИ SKILL-GAP АНАЛІЗУ ===
  Вибірка (вакансій з ≥2 скілами):     19,472
  Поріг "популярний" скіл (≥):         100 згадок
  Кількість популярних скілів:         203
  Поріг support пари:                  ≥ 0,5%
  Кількість пар у результатах:         951


In [14]:
# Таблиця 3.3 — топ пар за lift
top_pairs = pairs.sort_values('lift', ascending=False).head(15)

print('=== ТАБЛИЦЯ 3.3 — ТОП-15 ПАР ЗА LIFT ===')
print(f'{"A":<18}{"B":<18}{"supp%":>8}{"conf A→B":>10}{"conf B→A":>10}{"lift":>8}')
print('-' * 72)
for _, r in top_pairs.iterrows():
    print(f'{r["A"]:<18}{r["B"]:<18}{r["support_AB"]*100:>7.2f}%{r["conf_A_to_B"]*100:>9.1f}%{r["conf_B_to_A"]*100:>9.1f}%{r["lift"]:>8.2f}')

=== ТАБЛИЦЯ 3.3 — ТОП-15 ПАР ЗА LIFT ===
A                 B                    supp%  conf A→B  conf B→A    lift
------------------------------------------------------------------------
I2C               SPI                  0.72%     92.1%     90.9%  116.46
I2C               UART                 0.71%     91.4%     86.3%  110.60
SPI               UART                 0.72%     90.9%     87.0%  109.95
LangChain         LlamaIndex           0.57%     26.4%     91.0%   42.18
DNS               TCP/IP               0.55%     45.8%     47.4%   39.08
LangChain         LangGraph            0.63%     29.3%     75.9%   35.20
EC2               RDS                  0.63%     47.1%     45.2%   33.74
Cypress           Playwright           0.51%     63.7%     25.9%   32.13
Facebook Ads      Google Ads           0.96%     63.1%     46.7%   30.85
NumPy             Pandas               1.87%     86.1%     67.0%   30.76
PHP               Symfony              0.64%     19.4%     96.2%   29.03
CSS       

In [15]:
# Таблиця 3.4 — приклади рекомендацій
showcase_skills = ['React', 'Python', 'AWS', 'Docker', 'Kubernetes', 'TypeScript', 'Java']

print('=== ТАБЛИЦЯ 3.4 — SKILL-GAP РЕКОМЕНДАЦІЇ ===')
for skill in showcase_skills:
    sub = recs[recs['skill'] == skill].sort_values('confidence_pct', ascending=False).head(3)
    if not sub.empty:
        recs_str = ', '.join([f'{r["recommended"]} ({r["confidence_pct"]:.0f}%)' for _, r in sub.iterrows()])
        print(f'  {skill:<15} → {recs_str}')

=== ТАБЛИЦЯ 3.4 — SKILL-GAP РЕКОМЕНДАЦІЇ ===
  React           → TypeScript (48%), Node.js (40%), PostgreSQL (38%)
  Python          → AWS (33%), Docker (33%), Kubernetes (27%)
  AWS             → Python (57%), Docker (48%), Kubernetes (47%)
  Docker          → Python (56%), Kubernetes (55%), AWS (47%)
  Kubernetes      → Docker (57%), Python (47%), AWS (47%)
  TypeScript      → React (50%), Node.js (40%), JavaScript (37%)
  Java            → Kubernetes (37%), AWS (37%), Docker (36%)


## 9. Підсумкова таблиця всіх ключових цифр для звіту

In [16]:
print('=' * 70)
print('ПІДСУМКОВА ДОВІДКА ')
print('=' * 70)

print('\n[ДАНІ]')
print(f'  Сирих записів зібрано:              69 949')
print(f'  Період збору:                       05.03.2025 – 12.05.2026')
print(f'  Після точної агрегації (etap 1):    46 227')
print(f'  Унікальних батьківських (etap 2):   27 720')
print(f'  Дочірніх (зберігаються в кластері): 18 507')
print(f'  Зниження потужності:                60,4%')

print('\n[СКІЛИ]')
print(f'  Загальна кількість згадок:          199 959')
print(f'  Унікальних канонічних:              34 780')
print(f'  Вакансій з ≥1 скілом:               19 797 (71,4%)')
print(f'  Середня к-сть скілів/вакансія:      7,21')

print('\n[ЗАРПЛАТА]')
print(f'  Parents з salary_min:               4 498')
print(f'  Parents з обома (min і max):        4 497')
print(f'  Після фільтрації + Djinni-only:     2 618 (N для регресії)')

print('\n[РЕГРЕСІЯ]')
print(f'  N:                                  2 618')
print(f'  Скілів у моделі:                    48')
print(f'  R²:                                 0,5865')
print(f'  Adj R²:                             0,5743')
print(f'  F:                                  48,06 (p < 0,001)')
print(f'  Значущих ефектів (p<0,05):          9 (+6 / –3)')
print(f'  Топ позитивний:                     TensorFlow (+26,9%)')
print(f'  Топ негативний:                     Bash (–14,8%)')
print(f'  Ефект досвіду:                      +30,9% / рік')
print(f'  Ефект English A2:                   –20,8%')

print('\n[SKILL-GAP]')
print(f'  Вакансій з ≥2 скілами:              19 472')
print(f'  Популярних скілів (≥100):           203')
print(f'  Пар (support ≥ 0,5%):               951')
print(f'  Топ пара за lift:                   I2C ↔ SPI (lift 116,5)')
print(f'  Найкраща рекомендація для React:    TypeScript (48%)')
print(f'  Найкраща рекомендація для Python:   AWS (33%)')

print('\n' + '=' * 70)

ПІДСУМКОВА ДОВІДКА 

[ДАНІ]
  Сирих записів зібрано:              69 949
  Період збору:                       05.03.2025 – 12.05.2026
  Після точної агрегації (etap 1):    46 227
  Унікальних батьківських (etap 2):   27 720
  Дочірніх (зберігаються в кластері): 18 507
  Зниження потужності:                60,4%

[СКІЛИ]
  Загальна кількість згадок:          199 959
  Унікальних канонічних:              34 780
  Вакансій з ≥1 скілом:               19 797 (71,4%)
  Середня к-сть скілів/вакансія:      7,21

[ЗАРПЛАТА]
  Parents з salary_min:               4 498
  Parents з обома (min і max):        4 497
  Після фільтрації + Djinni-only:     2 618 (N для регресії)

[РЕГРЕСІЯ]
  N:                                  2 618
  Скілів у моделі:                    48
  R²:                                 0,5865
  Adj R²:                             0,5743
  F:                                  48,06 (p < 0,001)
  Значущих ефектів (p<0,05):          9 (+6 / –3)
  Топ позитивний:                   